# 2: Entity Resolution (Part 2) & Sample Construction

**Objective**
Combine the raw Orbis exports for both ETS and non-ETS universes into two clean, non-overlapping company samples (final_ets_sample.csv, final_non_ets_sample.csv) that meet the data availability and listing criteria required for the study period (2018–2024).

**Strategy**
The core challenge is twofold:
* correctly consolidating ETS installations under their **listed parent entities** using GUO logic, since installations are registered at the operating subsidiary level but stock/financial data exists at the listed parent level
* constructing a non-ETS control group that is free of carbon-regulated entities and comparable in terms of listing characteristics, so that any return differences can be attributed to carbon exposure rather than confounding factors like exchange, size, or listing era.


Filtering is applied programmatically and sequentially, with each stage's row count logged, so that attrition is auditable and defensible in the methodology write-up.

### Load Listing Dataframes and Schema Validation

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

In [2]:
# ── path setup ────────────────────────────────────────────────
sys.path.append(str(Path().resolve().parent))
from src.config import DATA_RAW, DATA_PROCESSED

# ── reusable loader ───────────────────────────────────────────
def load_processed(filename, **kwargs):
    """Load a processed CSV from data/raw folder."""
    return pd.read_csv(DATA_PROCESSED / filename, low_memory=False, **kwargs)

def load_raw(filename, **kwargs):
    """Load a raw CSV from data/raw folder."""
    return pd.read_csv(DATA_RAW / filename, low_memory=False, **kwargs)

# ── reusable profiler ─────────────────────────────────────────
def quick_profile(df, name="DataFrame"):
    """Print shape, dtypes, missing rates."""
    print(f"\n{'='*50}")
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"{'='*50}")

    print("\nColumns + dtypes:")
    print(df.dtypes.to_string())

    print("\nMissing value rates (%):")
    missing = (df.isnull().sum() / len(df) * 100).round(2)
    print(missing[missing > 0].to_string() if missing.any() else "  None")
    print()
    
    return df.head(3)


In [3]:
ets_listing = load_processed('ets_listing_processed.csv')
eu_listing = load_processed('eu_listing_processed.csv')

Some stages of schema validations:
* Deduplicate on bvd_id (Orbis exports sometimes have overlapping boundary rows across parts)
* Log row counts pre/post deduplication
* Confirm identical column schema across both DataFrames
* Standardise data types (dates as datetime, IDs as string to preserve leading zeros)
* Log missing value rates per column for documentation



In [4]:
def de_duplications(listing_df):
    df = listing_df.copy()
    pre_deduplication = len(df)
    print('Number of rows pre-deduplication: ',pre_deduplication)

    duplicated_rows = df[df.duplicated()]
    n_duplicated_rows = df.duplicated().sum()
    print('Number of duplicated rows: ',n_duplicated_rows)

    deduplicated_df = df.drop(duplicated_rows.index)
    print(f'Number of rows post-deduplication: {len(deduplicated_df)}\n')

    return deduplicated_df, duplicated_rows, n_duplicated_rows

In [5]:
ets_listing, ets_duplicated_rows, ets_n_duplicated_rows = de_duplications(ets_listing)
eu_listing, eu_duplicated_rows, eu_n_duplicated_rows = de_duplications(eu_listing)


Number of rows pre-deduplication:  7959
Number of duplicated rows:  1
Number of rows post-deduplication: 7958

Number of rows pre-deduplication:  21664
Number of duplicated rows:  4
Number of rows post-deduplication: 21660



In [6]:
# Confirm that columns and 
# ── reusable profiler ─────────────────────────────────────────
def quick_profile(df, name="DataFrame"):
    """Print shape, dtypes, missing rates."""
    print(f"\n{'='*50}")
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"{'='*50}")

    print("\nColumns + dtypes:")
    print(df.dtypes.to_string())

    print("\nMissing value rates (%):")
    missing = (df.isnull().sum() / len(df) * 100).round(2)
    print(missing[missing > 0].to_string() if missing.any() else "  None")
    print()
    
    return df.head(3)

In [7]:
def standardise_schema(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardise column names, data types, and string formatting.
    - Column names: lowercase, underscored
    - BvD IDs and string IDs: force to string, strip whitespace
    - Date columns: parse to datetime
    - Numeric columns: coerce where appropriate
    """
    # Standardise column names
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r'[\s/\-]+', '_', regex=True)
        .str.replace(r'[^\w]', '', regex=True)
    )

    # Define ID columns to keep as string
    id_cols = [c for c in df.columns if any(
        kw in c for kw in ['bvd_id', 'bvdid', 'isin', 'ticker',
                            'tax_id', 'trade_register', 'nace']
    )]
    for col in id_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().replace('nan', pd.NA)

    # Define date columns to parse
    date_cols = [c for c in df.columns if any(
        kw in c for kw in ['date', 'ipo', 'delisting', 'incorporation']
    )]
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)

    # Remaining object columns: strip whitespace
    obj_cols = df.select_dtypes(include='object').columns
    for col in obj_cols:
        df[col] = df[col].str.strip()

    return df


def validate_schema(df_ets: pd.DataFrame, df_non_ets: pd.DataFrame) -> bool:
    """
    Confirm identical column schema between ETS and non-ETS DataFrames.
    Logs any discrepancies clearly.
    """
    ets_cols = set(df_ets.columns)
    non_ets_cols = set(df_non_ets.columns)

    only_in_ets = ets_cols - non_ets_cols
    only_in_non_ets = non_ets_cols - ets_cols

    print(f"\n{'='*50}")
    print("Schema Validation")
    print(f"{'='*50}")
    print(f"ETS columns:     {len(ets_cols)}")
    print(f"Non-ETS columns: {len(non_ets_cols)}")

    if only_in_ets:
        print(f"\n  [!] Only in ETS:     {sorted(only_in_ets)}")
    if only_in_non_ets:
        print(f"  [!] Only in non-ETS: {sorted(only_in_non_ets)}")
    if not only_in_ets and not only_in_non_ets:
        print("\n  [OK] Schemas are identical.")
        return True

    return False

In [8]:
ets_listing     = standardise_schema(ets_listing)
eu_listing = standardise_schema(eu_listing)

validate_schema(ets_listing, eu_listing)


Schema Validation
ETS columns:     40
Non-ETS columns: 40

  [OK] Schemas are identical.


True

### Normalizing Listing Tables

In [9]:
# profiling listing dataframe and normalizing vat_tax_numbers:
quick_profile(ets_listing)


DataFrame: 7,958 rows × 40 cols

Columns + dtypes:
company_name                                               object
country_iso_code                                           object
nace_rev_2_core_code_4_digits                              object
consolidation_code                                         object
last_avail_year                                           float64
operating_revenue_turnover_th_usd_last_avail_yr            object
number_of_employees_last_avail_yr                          object
country                                                    object
website_address                                            object
bvd_sectors                                                object
bvd_id_number                                              object
trade_register_number                                      object
vat_tax_number                                             object
european_vat_number                                        object
tax_identification_numbe

,company_name,country_iso_code,nace_rev_2_core_code_4_digits,consolidation_code,last_avail_year,operating_revenue_turnover_th_usd_last_avail_yr,number_of_employees_last_avail_yr,country,website_address,bvd_sectors,...,duo_name,duo_ticker_symbol,duo_country_iso_code,ish_name,ish_bvd_id_number,ish_ticker_symbol,ish_country_iso_code,standardized_legal_form,nace_rev_2_core_code_description,nace_rev_2_core_code_4_digits1
0,BAYERISCHE MOTOREN WERKE AG,DE,2910.0,C2,2025.0,157701411.601067,154540,Germany,www.bmwgroup.com,Transport Manufacturing,...,BAYERISCHE MOTOREN WERKE AG,BMW,DE,NaN,<NA>,<NA>,NaN,Public limited companies,Manufacture of motor vehicles,2910.0
1,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,<NA>,NaN,NaN,<NA>,<NA>,NaN,NaN,<NA>,<NA>
2,ALLIANZ SE,DE,6511.0,C2,2025.0,120859295.571804,n.a.,Germany,www.allianz.com,"Banking, Insurance & Financial Services",...,ALLIANZ SE,ALV,DE,NaN,<NA>,<NA>,NaN,Public limited companies,Life insurance,6511.0


In [194]:
quick_profile(eu_listing)


DataFrame: 21,660 rows × 40 cols

Columns + dtypes:
company_name                                               object
country_iso_code                                           object
nace_rev_2_core_code_4_digits                              object
consolidation_code                                         object
last_avail_year                                           float64
operating_revenue_turnover_th_usd_last_avail_yr            object
number_of_employees_last_avail_yr                          object
country                                                    object
website_address                                            object
bvd_sectors                                                object
bvd_id_number                                              object
trade_register_number                                      object
vat_tax_number                                             object
european_vat_number                                        object
tax_identification_numb

,company_name,country_iso_code,nace_rev_2_core_code_4_digits,consolidation_code,last_avail_year,operating_revenue_turnover_th_usd_last_avail_yr,number_of_employees_last_avail_yr,country,website_address,bvd_sectors,...,duo_name,duo_ticker_symbol,duo_country_iso_code,ish_name,ish_bvd_id_number,ish_ticker_symbol,ish_country_iso_code,standardized_legal_form,nace_rev_2_core_code_description,nace_rev_2_core_code_4_digits1
0,SHELL PLC,GB,610.0,C1,2025.0,267418000.0,85000,United Kingdom,www.shell.com,Mining & Extraction,...,SHELL PLC,SHEL,GB,NaN,<NA>,<NA>,NaN,Public limited companies,Extraction of crude petroleum,610.0
1,GLENCORE PLC,GB,2399.0,C1,2025.0,247728000.0,n.a.,United Kingdom,www.glencore.com,"Leather, Stone, Clay & Glass products",...,GLENCORE PLC,GLEN,GB,NaN,<NA>,<NA>,NaN,Public limited companies,Manufacture of other non-metallic mineral prod...,2399.0
2,BP P.L.C.,GB,1920.0,C1,2025.0,189612000.0,93700,United Kingdom,www.bp.com,"Chemicals, Petroleum, Rubber & Plastic",...,BP P.L.C.,BP.,GB,NaN,<NA>,<NA>,NaN,Public limited companies,Manufacture of refined petroleum products,1920.0


Raw Orbis exports contain one row per company per multi-valued attribute (VAT numbers), resulting in continuation null rows for all other fields. Further, ownership data contains significant NULLs, showing consistent null rates across the same group of data.

 Missing values in the raw export are therefore structurally missing (MNAR) rather than reflecting genuine data absence. To eliminate structural missingness and produce an interpretable core entity table, we apply vertical decomposition. The raw DataFrame is split into:
* (1) a core entity table keyed on BvD ID
* (2) a long-format identifier table for tax and regulatory identifiers
* (3) a long-format ownership table capturing GUO, DUO, and ISH relationships. 

Missing values remaining in the core entity table after decomposition are genuinely missing and treated as such in subsequent analysis.

In [22]:
def decompose_core(df: pd.DataFrame, id_col: str = 'bvd_id_number') -> pd.DataFrame:
    """
    Extract core entity table — one row per company.
    Drops continuation null rows (where bvd_id is null) and all
    identifier/ownership columns that are decomposed into separate tables.
    Remaining missingness in this table is genuine, not structural.
    """
    
    id_col = 'bvd_id_number'
    
    #isolate identifier columns & Ownship columns:
    identifier_cols = [
        'vat_tax_number', 'european_vat_number',
        'tax_identification_number_tin', 'lei_legal_entity_identifier',
        'trade_register_number'
    ]
    ownership_cols = [
        'guo_name', 'guo_bvd_id_number', 'guo_country_iso_code', 'guo_ticker_symbol',
        'duo_name', 'duo_bvd_id_number', 'duo_country_iso_code', 'duo_ticker_symbol',
        'ish_name', 'ish_bvd_id_number', 'ish_country_iso_code', 'ish_ticker_symbol'
    ]

    cols_to_drop = [c for c in identifier_cols + ownership_cols if c in df.columns]

    # Drop non-core columns for core df:
    core = (
        df.dropna(subset=[id_col])
          .drop(columns=cols_to_drop)
          .drop_duplicates(subset=[id_col])
          .reset_index(drop=True)
    )

    print(f"\n{'='*50}")
    print("Core Entity Table")
    print(f"{'='*50}")
    
    print(f"  Rows: {len(core):,}")
    print(f"  Columns: {len(core.columns)}")
    print(f"\n  Remaining missing rates (%):")
    missing = (core.isnull().sum() / len(core) * 100).round(2)
    print(missing[missing > 0].to_string() if missing.any() else "  None")

    return core




In [24]:
def decompose_identifiers(df: pd.DataFrame,
                           id_col: str = 'bvd_id_number') -> pd.DataFrame:
    """
    Extract long-format identifier table (one row per BvD ID + identifier type).
    Covers: VAT, European VAT, TIN, LEI, trade register number.
    Rows where identifier_value is null are dropped.
    """

    id_col = 'bvd_id_number'

    # map existing identifier columns in listing table:
    identifier_cols = {
        'vat_tax_number':               'vat_tax_number',
        'european_vat_number':          'european_vat_number',
        'tax_identification_number_tin':'tin', # shorten
        'lei_legal_entity_identifier':  'lei', # shorten
        'trade_register_number':        'trade_register_number'
    }

    # Only use columns that exist in df (contain in available dictionary)
    available = {k: v for k, v in identifier_cols.items() if k in df.columns}

    # Convert wide listing table to long identifier table:
    frames = []
    for col, label in available.items():
        tmp = (
            df[[id_col, col]]
            .dropna(subset=[id_col])   # drop continuation null rows first
            .dropna(subset=[col])      # then drop missing identifiers
            .rename(columns={col: 'identifier_value'})
            .assign(identifier_type=label)
        )
        frames.append(tmp)

    identifier_df = (
        pd.concat(frames, ignore_index=True)
          [[id_col, 'identifier_type', 'identifier_value']]
          .drop_duplicates()
          .reset_index(drop=True)
    )

    print(f"\n{'='*50}")
    print("Identifier Table")
    print(f"{'='*50}")
    print(f"  Total rows:              {len(identifier_df):,}")
    print(f"  Unique BvD IDs:          {identifier_df[id_col].nunique():,}")
    print(f"\n  Rows per identifier type:")
    print(identifier_df['identifier_type'].value_counts().to_string())

    return identifier_df



In [43]:
def decompose_ownership(df: pd.DataFrame,
                        id_col: str = 'bvd_id_number') -> pd.DataFrame:
    """
    Extract long-format ownership table (one row per BvD ID + owner type).
    Covers: GUO, DUO, ISH.
    Rows where all owner fields are null are dropped.
    """
    id_col = 'bvd_id_number'
    
    ownership_groups = {
        'GUO': {
            'name':       'guo_name',
            'bvd_id':     'guo_bvd_id_number',
            'country':    'guo_country_iso_code',
            'ticker':     'guo_ticker_symbol'
        },
        'DUO': {
            'name':       'duo_name',
            'bvd_id':     'duo_bvd_id_number',
            'country':    'duo_country_iso_code',
            'ticker':     'duo_ticker_symbol'
        },
        'ISH': {
            'name':       'ish_name',
            'bvd_id':     'ish_bvd_id_number',
            'country':    'ish_country_iso_code',
            'ticker':     'ish_ticker_symbol'
        }
    }
    
    
    # convert 
    frames = [] # initiate temp ownership

    # loop through each owner type and each column within:
    for owner_type, col_map in ownership_groups.items():
        # Only use columns present in df
        available = {k: v for k, v in col_map.items() if v in df.columns}
        if not available:
            continue
        src_cols = list(available.values())

        rename_map = {v: k for k, v in available.items()}

    # convert each owner-type & column type (wide) to long temp table
        tmp = (
            df[[id_col] + src_cols]
            .dropna(subset=[id_col])          # drop continuation null rows
            .dropna(subset=src_cols, how='all') # drop rows where all owner fields null
            .rename(columns=rename_map)
            .assign(owner_type=owner_type)
        )
        frames.append(tmp)# and append to list

    # 
    ownership_df = (
        pd.concat(frames, ignore_index=True) # create dataframe
          [[id_col, 'owner_type', 'name', 'bvd_id', 'country', 'ticker']]
          .rename(columns={
              'name':   'owner_name',
              'bvd_id': 'owner_bvd_id_number',
              'country':'owner_country',
              'ticker': 'owner_ticker'
          })
          .drop_duplicates()
          .reset_index(drop=True)
    )

    print(f"\n{'='*50}")
    print("Ownership Table")
    print(f"{'='*50}")
    print(f"  Total rows:     {len(ownership_df):,}")
    # print(f"  Unique BvD IDs: {len(ownership_df[id_col].unique()):,}")
    print(f"\n  Rows per owner type:")
    print(ownership_df['owner_type'].value_counts().to_string())

    return ownership_df

In [44]:
# ETS
ets_core       = decompose_core(ets_listing)
ets_identifiers = decompose_identifiers(ets_listing)
ets_ownership  = decompose_ownership(ets_listing)


Core Entity Table
  Rows: 6,803
  Columns: 23

  Remaining missing rates (%):
country_iso_code                     0.01
nace_rev_2_core_code_4_digits        1.59
last_avail_year                      2.25
country                              0.01
website_address                      8.25
bvd_sectors                          2.12
ticker_symbol                       95.35
isin_number                         95.27
date_of_incorporation               11.95
currency                            96.46
type_of_share                       96.46
ipo_date                            97.00
delisting_date                      97.68
nace_rev_2_core_code_description     1.59
nace_rev_2_core_code_4_digits1       1.59

Identifier Table
  Total rows:              25,253
  Unique BvD IDs:          6,789

  Rows per identifier type:
identifier_type
european_vat_number      5890
vat_tax_number           5205
lei                      4984
trade_register_number    4975
tin                      4199

Ownership 

In [39]:
# Non-ETS
eu_core        = decompose_core(eu_listing)
eu_identifiers = decompose_identifiers(eu_listing)
eu_ownership   = decompose_ownership(eu_listing)


Core Entity Table
  Rows: 20,020
  Columns: 23

  Remaining missing rates (%):
nace_rev_2_core_code_4_digits        1.39
last_avail_year                     22.77
website_address                      8.72
bvd_sectors                          1.53
ticker_symbol                       24.53
isin_number                         23.63
date_of_incorporation               58.62
currency                            38.34
type_of_share                       38.34
main_exchange                       19.70
ipo_date                            42.85
delisting_date                      66.20
standardized_legal_form              0.03
nace_rev_2_core_code_description     1.39
nace_rev_2_core_code_4_digits1       1.39

Identifier Table
  Total rows:              55,953
  Unique BvD IDs:          19,200

  Rows per identifier type:
identifier_type
lei                      16305
trade_register_number    11857
european_vat_number      10367
vat_tax_number            9173
tin                       8251

Own

In [48]:
def verify_roundtrip(df_original: pd.DataFrame,
                     core: pd.DataFrame,
                     identifiers: pd.DataFrame,
                     ownership: pd.DataFrame,
                     id_col: str = 'bvd_id_number',
                     name: str = "DataFrame") -> None:
    """
    Verify decomposition roundtrip by rejoining core, identifiers, and ownership
    and comparing shape and values against the original cleaned DataFrame.
    
    Note: compare against the post-standardise, post-dropna(bvd_id) original,
    not the raw export — continuation null rows are intentionally removed.
    """
    id_col = 'bvd_id_number'
    

    # (1) Pivot long identifiers dataframe back to wide:
    id_wide = (
        identifiers
        .pivot_table(index=id_col,
                     columns='identifier_type',
                     values='identifier_value',
                     aggfunc='first')
        .reset_index()
    )
    id_wide.columns.name = None

    # (2) Pivot long ownership dataframe back to wide:
    frames = []
    for owner_type in ownership['owner_type'].unique():
        tmp = (
            ownership[ownership['owner_type'] == owner_type] # subset dataframe for each ownership type
            .drop(columns='owner_type')
            .rename(columns={
                'owner_name':    f'{owner_type.lower()}_name',
                'owner_bvd_id':  f'{owner_type.lower()}_bvd_id_number',
                'owner_country': f'{owner_type.lower()}_country_iso_code',
                'owner_ticker':  f'{owner_type.lower()}_ticker_symbol'
            })
        )
        frames.append(tmp)

    # merge each appended tmp horizontally (outer) on id_col:
    own_wide = frames[0]
    for f in frames[1:]:
        own_wide = own_wide.merge(f, on=id_col, how='outer')

    # Rejoin everything
    reconstructed = (
        core
        .merge(id_wide,  on=id_col, how='left')
        .merge(own_wide, on=id_col, how='left')
    )

    # --- Checks ---
    print(f"\n{'='*50}")
    print(f"Roundtrip Verification — {name}")
    print(f"{'='*50}")

    # 1. Row count
    # Original must be filtered to non-null bvd_id rows first (continuation rows removed)
    df_clean = df_original.dropna(subset=[id_col]).drop_duplicates(subset=[id_col])
    print(f"\n  Row counts:")
    print(f"    Original (post-clean): {len(df_clean):,}")
    print(f"    Reconstructed:         {len(reconstructed):,}")
    print(f"    Match: {'YES' if len(df_clean) == len(reconstructed) else 'NO — investigate'}")

    # 2. BvD ID sets match
    orig_ids  = set(df_clean[id_col])
    recon_ids = set(reconstructed[id_col])
    missing_from_recon = orig_ids - recon_ids
    extra_in_recon     = recon_ids - orig_ids
    print(f"\n  BvD ID set:")
    print(f"    Missing from reconstructed: {len(missing_from_recon)}")
    print(f"    Extra in reconstructed:     {len(extra_in_recon)}")

    # 3. Column coverage
    orig_cols  = set(df_clean.columns)
    recon_cols = set(reconstructed.columns)
    print(f"\n  Column coverage:")
    print(f"    Original:      {len(orig_cols)}")
    print(f"    Reconstructed: {len(recon_cols)}")
    missing_cols = orig_cols - recon_cols
    if missing_cols:
        print(f"    Missing cols:  {sorted(missing_cols)}")
    else:
        print(f"    All original columns present: YES")

    # 4. Spot-check nulls match on core columns
    core_cols = list(core.columns)
    core_cols.remove(id_col)
    orig_nulls  = df_clean[core_cols].isnull().sum()
    recon_nulls = reconstructed[core_cols].isnull().sum()
    mismatch = orig_nulls[orig_nulls != recon_nulls]
    print(f"\n  Null count mismatches on core columns:")
    if mismatch.empty:
        print(f"    None — core columns match exactly")
    else:
        print(mismatch.to_string())

In [49]:
verify_roundtrip(ets_listing,ets_core, ets_identifiers, ets_ownership, name='ETS')


Roundtrip Verification — ETS

  Row counts:
    Original (post-clean): 6,803
    Reconstructed:         6,803
    Match: YES

  BvD ID set:
    Missing from reconstructed: 0
    Extra in reconstructed:     0

  Column coverage:
    Original:      40
    Reconstructed: 40
    Missing cols:  ['duo_bvd_id_number', 'guo_bvd_id_number', 'ish_bvd_id_number', 'lei_legal_entity_identifier', 'tax_identification_number_tin']

  Null count mismatches on core columns:
    None — core columns match exactly


In [45]:
verify_roundtrip(eu_listing, eu_core, eu_identifiers,eu_ownership, name = 'EU')


Roundtrip Verification — EU

  Row counts:
    Original (post-clean): 20,020
    Reconstructed:         20,020
    Match: YES

  BvD ID set:
    Missing from reconstructed: 0
    Extra in reconstructed:     0

  Column coverage:
    Original:      40
    Reconstructed: 40
    Missing cols:  ['duo_bvd_id_number', 'guo_bvd_id_number', 'ish_bvd_id_number', 'lei_legal_entity_identifier', 'tax_identification_number_tin']

  Null count mismatches on core columns:
    None — core columns match exactly


### (2) Mapping Parent Companies Tickers to S&P

Case 1: Entity IS its own GUO (bvd_id == owner_bvd_id)
* already the listed parent (BMW row 0, Allianz row 1, ENI row 3)
* use directly

Case 2: Entity has a GUO with a ticker (listing):
* unlisted subsidiary of a listed parent
*  assign parent's bvd_id and ticker

Case 3: Entity has a GUO but no ticker (row 2 - Government of Norway, row 4 - Bosch)
* parent is unlisted/state-owned
* flag as unresolvable, exclude from ETS listed sample

Case 4: No GUO recorded
* 6803 - 5684 = 1,119 entities with no ownership data (majority are unlisted/delisted)
* check identifiers table for LEI/ticker as fallback
* otherwise flag as unresolvable


Further, access to Orbis is now not possible; link GUO listing status via S&P Capital IQ Pro

In [139]:
ets_owner_ticker['bare_ticker'] = ets_ownership['owner_ticker'].str.split('.').str[0]

/var/folders/bg/1y0v_kw15cnb9w8c866jb_280000gn/T/ipykernel_28764/212070194.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ets_owner_ticker['bare_ticker'] = ets_ownership['owner_ticker'].str.split('.').str[0]


In [189]:
# Get all the ownership tickers
ets_self_owned = ets_ownership[ets_ownership['bvd_id_number'] == ets_ownership['owner_bvd_id_number']]
ets_non_selfowned = ets_ownership[ets_ownership['bvd_id_number'] != ets_ownership['owner_bvd_id_number']]


# Get all the non-self-owned companies with tickers:
ets_owner_ticker = ets_non_selfowned[
    ets_non_selfowned['owner_ticker'].notna() &
    ~ets_non_selfowned['owner_ticker'].isin(['-', 'Delisted', ''])
]

ets_ticker_lookup = (
    ets_owner_ticker[['owner_ticker', 'owner_name']]
    .drop_duplicates(subset=['owner_ticker'])
    .reset_index(drop=True)
)

# Generate all variants of tickers:
ets_ticker_lookup['bare_ticker'] = ets_ticker_lookup['owner_ticker'].str.split('.').str[0]
ets_ticker_lookup['space_ticker'] = ets_ticker_lookup['owner_ticker'].str.replace('.', ' ', regex=False)
combined_ets_ticker_lookup = pd.concat([ets_ticker_lookup['bare_ticker'],
                                        ets_ticker_lookup['space_ticker'],
                                        ets_ticker_lookup['owner_ticker']], ignore_index=True).to_frame(name='ticker')
combined_ets_ticker_lookup = combined_ets_ticker_lookup.drop_duplicates()

# Get all the delisted owners with no tickers (why? if delisting is after study year of 2018, still need to consider):
ets_owner_delisted_name = ets_non_selfowned[
    ets_non_selfowned['owner_ticker'] == 'Delisted'
]['owner_name'].unique()

ets_delisted_lookup = (
    ets_non_selfowned[ets_non_selfowned['owner_ticker'] == 'Delisted']
    [['owner_name', 'owner_bvd_id_number', 'owner_country']]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [538]:
# Ground truth: ets_listing
print(f"Total unique ETS entities (ets_listing): {ets_listing['bvd_id_number'].nunique():,}")

# Cross-reference with ets_ownership
entities_with_ownership = ets_ownership['bvd_id_number'].nunique()
entities_without_ownership = ets_listing[
    ~ets_listing['bvd_id_number'].isin(ets_ownership['bvd_id_number'])
]['bvd_id_number'].nunique()

print(f"Entities WITH ownership record:    {entities_with_ownership:,}")
print(f"Entities WITHOUT ownership record: {entities_without_ownership:,}")
print(f"Sum:                               {entities_with_ownership + entities_without_ownership:,}")

# For entities with ownership — break down by listing_status in ets_listing
has_ownership = ets_listing[ets_listing['bvd_id_number'].isin(ets_ownership['bvd_id_number'])]
print(f"\nEntities WITH ownership, by listing_status:")
print(has_ownership['listing_status'].value_counts(dropna=False).to_string())

# For entities without ownership — break down by listing_status
no_ownership = ets_listing[~ets_listing['bvd_id_number'].isin(ets_ownership['bvd_id_number'])]
print(f"\nEntities WITHOUT ownership, by listing_status:")
print(no_ownership['listing_status'].value_counts(dropna=False).to_string())

# Self-owned vs non-self-owned (from ets_ownership, unique entities only)
self_owned_bvds = ets_ownership[
    ets_ownership['bvd_id_number'] == ets_ownership['owner_bvd_id_number']
]['bvd_id_number'].unique()

non_self_owned_bvds = ets_ownership[
    ets_ownership['bvd_id_number'] != ets_ownership['owner_bvd_id_number']
]['bvd_id_number'].unique()

print(f"\nSelf-owned entities (unique bvd_ids):     {len(self_owned_bvds):,}")
print(f"Non-self-owned entities (unique bvd_ids): {len(non_self_owned_bvds):,}")
print(f"Overlap (both self and non-self):         "
      f"{len(set(self_owned_bvds) & set(non_self_owned_bvds)):,}")

# Breakdown of non-self-owned by listing_status and ticker availability
non_self_owned_df = ets_listing[
    ets_listing['bvd_id_number'].isin(non_self_owned_bvds)
]
print(f"\nNon-self-owned entities, by listing_status:")
print(non_self_owned_df['listing_status'].value_counts(dropna=False).to_string())

Total unique ETS entities (ets_listing): 6,803
Entities WITH ownership record:    5,684
Entities WITHOUT ownership record: 1,119
Sum:                               6,803

Entities WITH ownership, by listing_status:
listing_status
Unlisted    5383
Delisted     175
Listed       126

Entities WITHOUT ownership, by listing_status:
listing_status
NaN         1155
Unlisted    1094
Delisted      21
Listed         4

Self-owned entities (unique bvd_ids):     1,769
Non-self-owned entities (unique bvd_ids): 5,178
Overlap (both self and non-self):         1,263

Non-self-owned entities, by listing_status:
listing_status
Unlisted    4955
Delisted     162
Listed        61


In [192]:
# Export for Capital IQ manual lookup
combined_ets_ticker_lookup.to_csv('ciq_ticker_lookup.csv', index=False)
ets_delisted_lookup.to_csv('ciq_delisted_lookup.csv', index=False)

In [390]:
ets_self_owned_full_listing = ets_listing[ets_listing['bvd_id_number'].isin(ets_self_owned['bvd_id_number'])]
ets_self_owned_full_listing[ets_self_owned_full_listing['listing_status'] == 'Unlisted']['ticker_symbol'].nunique()

0

In [270]:
ciq_tickers = load_raw('ciq_ets_owners_tickers_fetched.csv')

{'ELUX.B', 'RY8', '161390', 'NCC.B', 'PEAB', 'BRKB', 'PEAB.B', 'CZG', 'MOTHERSUMI', 'UU ', 'TATACHEM', 'RUAL', 'RR ', 'SKF.B', 'JBFIND', 'STW5', '02269', 'BA ', '051910', 'ENPG', 'TMPV', 'HARB', 'VOLCAR', 'BINANIIND', 'RO', '034730', 'VOLCAR.B', 'VNTRF', 'VOLV', 'SSAB.A', 'SSAB', 'SCA.B', 'ELUX', 'HOLM', 'SEB.A', 'ESSITY', 'LUND.B', 'ESSITY A', 'LBTYA', 'TATASTEEL', 'CTS1L', 'NEMAKA', '361610', 'HARB.B', 'ETEX', 'BP', 'LUND', 'SFP1', '373220', '000210', '005380', 'HOLM.B', 'ESSITY.A', '01113', 'UU', 'HINDALCO', 'TATYY', 'VOLV.B', 'ALPEKA', 'ROCK.B', '000270', 'SKA.B', 'BP '}
{nan, 'VMVI.X', 'VMVA.X', 'LYRIO', 'VINA.X', 'SPYR'}


Some variations of tickers fetched, in sequence:
* original ticker in ets_owner_ticker (exact matches first) (e.g., 'HOLM.B')
* ticker with suffix with '.' replace with ' ' (second pass of matching) (e.g., 'HOLM B')
* bare ticker with all suffixes removed (e.g., 'HOLM)

In [259]:
def match_ciq_tickers(ets_owner_ticker: pd.DataFrame,
                      ciq_tickers: pd.DataFrame) -> pd.DataFrame:
    """
    Match ETS owner tickers against S&P Capital IQ ticker data.
    Match key: bare_ticker + owner_country == SP_TICKER + SP_COUNTRY_CODE
    
    Returns merged DataFrame with:
    - All original ETS owner columns
    - Matched CIQ columns (SP_ENTITY_NAME, SP_TICKER, SP_EXCHANGE, SP_ISIN,
      SP_LEI, SP_COMPANY_TYPE, SP_COMPANY_STATUS, SP_COUNTRY_CODE, SP_IPO_DATE)
    - match_status flag
    """

    # Prepare CIQ side — keep only relevant columns, strip whitespace
    ciq_cols = [
        'SP_ENTITY_NAME', 'SP_ENTITY_ID', 'SP_TICKER', 'SP_EXCHANGE',
        'SP_ISIN', 'SP_LEI', 'SP_COMPANY_TYPE', 'SP_COMPANY_STATUS',
        'SP_COUNTRY_CODE', 'SP_IPO_DATE', 'SP_DATE_INCORPORATED'
    ]
    ciq = ciq_tickers[ciq_cols].copy()
    ciq['SP_TICKER']       = ciq['SP_TICKER'].astype(str)
    ciq['SP_COUNTRY_CODE'] = ciq['SP_COUNTRY_CODE'].astype(str).str.strip().str.upper()

    # Prepare ETS side to match all possible variations of tickers in ciq:
    ets = ets_owner_ticker.copy()
    ets['space_ticker']   = ets['owner_ticker'].str.replace('.',' ')
    ets['bare_ticker']    = ets['owner_ticker'].str.split('.').str[0]
    ets['owner_country']  = ets['owner_country'].astype(str).str.strip().str.upper()

    # --- Pass 1: Match on owner_ticker + country ---
    merged = ets.merge(
        ciq,
        left_on=['owner_ticker', 'owner_country'],
        right_on=['SP_TICKER', 'SP_COUNTRY_CODE'],
        how='left', # keep ets as the baselines
        indicator=True # creates a '_merged' column
    )
    
    # indicating merging status:
    merged['matching_status'] = merged['_merge'].map({
        'both':      'matched_base_ticker_country', # matched rows
        'left_only': 'unmatched' #unmatched rows
    })

    merged = merged.drop(columns='_merge')
    pass_1_merged = merged[merged['matching_status'] == 'matched_base_ticker_country']


    # --- Pass 2: Match on space_ticker ---
    pass_2_unmatched = merged[merged['matching_status'] == 'unmatched'].drop(columns=['matching_status'])
    pass_2_unmatched = pass_2_unmatched.drop(columns=ciq_cols)
    
    pass_2_merged = pass_2_unmatched.merge(
        ciq,
        left_on=['space_ticker'], # high specificity
        right_on=['SP_TICKER'],
        how='left',
        indicator=True
    )

    pass_2_merged['matching_status'] = pass_2_merged['_merge'].map({
        'both':      'space_ticker_only',
        'left_only': 'unmatched'
    })
    pass_2_merged = pass_2_merged.drop(columns='_merge')
    pass_3_unmatched = pass_2_merged[pass_2_merged['matching_status'] == 'unmatched'].drop(columns=['matching_status'])
    pass_3_unmatched = pass_3_unmatched.drop(columns=ciq_cols)
    
    pass_2_merged = pass_2_merged[pass_2_merged['matching_status'] == 'space_ticker_only']


    # --- Pass 3: Match on bare_ticker + country ---
    pass_3_merged = pass_3_unmatched.merge(
        ciq,
        left_on=['bare_ticker'],
        right_on='SP_TICKER',
        how='left',
        indicator=True
    )
    pass_3_merged['matching_status'] = pass_3_merged['_merge'].map({
        'both':      'bare_ticker_only',
        'left_only': 'unmatched'
    })
    pass_3_merged = pass_3_merged.drop(columns='_merge')

    final_merged_df = pd.concat([pass_1_merged,
                                 pass_2_merged,
                                 pass_3_merged], ignore_index=True)

    # --- Diagnostics ---
    print(f"\n{'='*50}")
    print("CIQ Ticker Match Results")
    print(f"{'='*50}")

    print(f"  Total ETS owner rows:          {len(ets):,}")
    print(f"\n  Match status breakdown:")
    print(final_merged_df['matching_status'].value_counts().to_string())
    print(f"\n  Unique tickers — input:   {ets['bare_ticker'].nunique():,}")
    print(f"  Unique tickers — matched: "
          f"{final_merged_df[final_merged_df['matching_status'] != 'unmatched']['bare_ticker'].nunique():,}")

    return final_merged_df

In [260]:
ets_owner_merged_ticker = match_ciq_tickers(ets_owner_ticker, ciq_tickers)


CIQ Ticker Match Results
  Total ETS owner rows:          2,329

  Match status breakdown:
matching_status
matched_base_ticker_country    2101
space_ticker_only               232
unmatched                       101

  Unique tickers — input:   529
  Unique tickers — matched: 496


In [274]:
# Check which tickers are not retrieved from CIQ:
ciq_ticker_set = set(ciq_tickers['SP_TICKER'].unique())
lookup_ticker_set = set(combined_ets_ticker_lookup['ticker'])


# Check whether these contributed to unmatched tickers:
unmatched_ets_ticker = set(ets_owner_merged_ticker[ets_owner_merged_ticker['matching_status'] == 'unmatched']['owner_ticker'].unique())
unretrieved_unmatched_ticker = unmatched_ets_ticker.intersection()

print(f"Ticker variants in lookup not fetched in CIQ: {lookup_ticker_set - ciq_ticker_set}")
print(f"Ticker in fetched in CIQ not matching any variants: {ciq_ticker_set - lookup_ticker_set}")
print(f"Unmatched tickers in the unfetched tickers CIQ: {unretrieved_unmatched_ticker} (n = {len(unretrieved_unmatched_ticker)})")



Ticker variants in lookup not fetched in CIQ: {'ELUX.B', 'RY8', '161390', 'NCC.B', 'PEAB', 'BRKB', 'PEAB.B', 'CZG', 'MOTHERSUMI', 'UU ', 'TATACHEM', 'RUAL', 'RR ', 'SKF.B', 'JBFIND', 'STW5', '02269', 'BA ', '051910', 'ENPG', 'TMPV', 'HARB', 'VOLCAR', 'BINANIIND', 'RO', '034730', 'VOLCAR.B', 'VNTRF', 'VOLV', 'SSAB.A', 'SSAB', 'SCA.B', 'ELUX', 'HOLM', 'SEB.A', 'ESSITY', 'LUND.B', 'ESSITY A', 'LBTYA', 'TATASTEEL', 'CTS1L', 'NEMAKA', '361610', 'HARB.B', 'ETEX', 'BP', 'LUND', 'SFP1', '373220', '000210', '005380', 'HOLM.B', 'ESSITY.A', '01113', 'UU', 'HINDALCO', 'TATYY', 'VOLV.B', 'ALPEKA', 'ROCK.B', '000270', 'SKA.B', 'BP '}
Ticker in fetched in CIQ not matching any variants: {nan, 'VMVI.X', 'VMVA.X', 'LYRIO', 'VINA.X', 'SPYR'}
Unmatched tickers in the unfetched tickers CIQ: {'RY8', '161390', 'BRKB', 'CZG', '02269', 'MOTHERSUMI', 'TATACHEM', 'RUAL', 'JBFIND', 'STW5', '051910', 'ENPG', 'TMPV', 'BINANIIND', 'RO', '034730', 'VNTRF', 'LBTYA', 'TATASTEEL', 'CTS1L', 'NEMAKA', '361610', 'ETEX', 'S

In [338]:
def resolve_unmatched_tickers(unmatched_df: pd.DataFrame,
                               supplement_df: pd.DataFrame,
                               ticker_remap: dict,
                               non_european: set) -> pd.DataFrame:
    """
    Resolve remaining unmatched tickers:
    - Remap confirmed ticker variants → match against supplement
    - Flag non-European parents
    - Flag remainder as unresolvable
    """
    # Load supplement
    supp = supplement_df.copy()
    supp['SP_TICKER'] = supp['SP_TICKER'].astype(str).str.strip()

    results = []

    for idx, row in unmatched_df.iterrows():
        ticker = row['bare_ticker']
        matched_row = row.to_dict()

        # Remapped tickers — look up corrected ticker in supplement
        if ticker in ticker_remap:
            correct_ticker = ticker_remap[ticker][0]
            supp_match = supp[supp['SP_TICKER'] == correct_ticker]
            if len(supp_match) > 0:
                matched_row.update(supp_match.iloc[0].to_dict())
                matched_row['matching_status'] = 'remapped_supplement'
            else:
                matched_row['matching_status'] = 'remapped_no_match'

        # Non-European parents
        elif ticker in non_european:
            matched_row['matching_status'] = 'non_european_parent'

        # Genuinely unresolvable
        else:
            matched_row['matching_status'] = 'unresolvable'

        results.append(matched_row)

    return pd.DataFrame(results)

In [339]:
# Ticker remapping — ticker in Orbis differs from CIQ ticker
TICKER_REMAP = {
    'CZG':      ('COLT',      'SEP',    'CZ'),
    'RO':       ('ROP',       'SWX',    'CH'),
    'ETEX':     ('094124453', 'ENXTBR', 'BE'),
    'RY8':      ('RY8',       'DUSE',   'DE'),   # ticker same, exchange found
    'CTS1L':    ('CTS',       'WSE',    'LT'),
    'ESSITY.A': ('ESSITY B',  'OM',     'SE'),
    'STW5':     ('ACT',       'XTRA',   'DE'),
}

# Non-European — flag and exclude from portfolio universe
NON_EUROPEAN_TICKERS = {
    # Already identified
    'TATACHEM', 'TATASTEEL', 'TATYY', 'HINDALCO', 'JBFIND',
    'MOTHERSUMI', 'RUAL', 'ENPG', 'BRKB', 'LBTYA', 'VNTRF',
    '161390', '051910', '034730', '373220', '000210',
    '005380', '000270', '361610', '02269', '01113',
    
    # Confirmed from manual check
    'NEMAKA', 'BINANIIND', 'TMPV', 'ALPEKA',
}

In [344]:

# Re-run on both unmatched and remapped_no_ciq_row
unmatched_ets_owner_merged_ticker = ets_owner_merged_ticker[ets_owner_merged_ticker['matching_status'] == 'unmatched'].copy()
ets_supplement_df = load_raw('ets_owner_supplement.csv')

resolved = resolve_unmatched_tickers(
    unmatched_ets_owner_merged_ticker,
    ets_supplement_df,
    ticker_remap=TICKER_REMAP,
    non_european=NON_EUROPEAN_TICKERS
)

cols_to_keep = unmatched_ets_owner_merged_ticker.columns
resolved = resolved[cols_to_keep]

final_merged_df = pd.concat( [ets_owner_merged_ticker[ets_owner_merged_ticker['matching_status'] != 'unmatched'], resolved], ignore_index=True)

print(final_merged_df['matching_status'].value_counts().to_string())

matching_status
matched_base_ticker_country    2101
space_ticker_only               232
non_european_parent              52
remapped_supplement              32
unresolvable                     17


In [348]:
# What owner_types are represented?
print(final_merged_df['owner_type'].value_counts())

# How many unique parents (by SP_ISIN / SP_TICKER)?
matched = final_merged_df[
    ~final_merged_df['matching_status'].isin(['non_european_parent', 'unresolvable'])
]

print(f"\nUnique parent ISINs:   {matched['SP_ISIN'].nunique():,}")
print(f"Unique parent tickers: {matched['SP_TICKER'].nunique():,}")

# How many unique ETS entities mapped to a resolved parent?
print(f"Unique ETS bvd_ids with resolved parent: {matched['bvd_id_number'].nunique():,}")
print(f"Unique ETS bvd_ids (starting with parents): {ets_owner_ticker['bvd_id_number'].nunique():,}")

# Multiple entities under same parent
parent_counts = (
    matched.groupby('SP_TICKER')['bvd_id_number']
    .nunique()
    .sort_values(ascending=False)
)
print(f"\nParents with >1 ETS entity:")
print(parent_counts[parent_counts > 1].head(10))

owner_type
GUO    1575
ISH     432
DUO     427
Name: count, dtype: int64

Unique parent ISINs:   537
Unique parent tickers: 502
Unique ETS bvd_ids with resolved parent: 1,537
Unique ETS bvd_ids (starting with parents): 1,596

Parents with >1 ETS entity:
SP_TICKER
ENGI    62
SGO     41
VIE     39
WIE     26
EOAN    26
MT      21
TTE     20
SW      18
HOLN    17
PKN     16
Name: bvd_id_number, dtype: int64


### (3) ETS Entity Consolidation (Parent Resolution)

### (3) Remove ETS Overlap from Non-ETS Universe



**Proposed Schema**

orbis_core          — keyed on (bvd_id, universe)

orbis_identifiers   — keyed on (bvd_id, identifier_type, universe)

orbis_ownership     — keyed on (bvd_id, owner_type, universe)

master_company_list — to be built

prices                  — to be built — keyed on (bvd_id, date)

fundamentals            — to be built — keyed on (bvd_id, year)

emissions               — to be built — keyed on (bvd_id, year)

signals                 — to be built — keyed on (bvd_id, date)

### (4) Filtering Companies

### (5) Exchange Whitelist & Stock Tickers Creation

### (6) Check Availability

Geography

Country of incorporation: EU27 + UK + Norway + Switzerland (keeps your "European equity markets" framing clean and matches EUA price exposure geography)
Listed on a major European exchange — use a whitelist approach (Euronext, LSE, Deutsche Börse, SIX, Nasdaq Nordic, Borsa Italiana, BME, etc.) rather than just country, since some European firms list primarily in the US

Listing status & data availability

Active OR delisted-after-2018 (same survivorship bias logic you applied to ETS accounts — include firms that were listed during your study period even if now gone)
IPO date before 2021 (gives at least 3 years of data within your 2018–2024 window)
Primary listing only — exclude secondary listings and depositary receipts to avoid duplicates

Size & liquidity

Exclude penny stocks: minimum share price threshold (e.g. >€0.50 average) or minimum market cap (e.g. >€50m) — exact cutoff is somewhat arbitrary, just document it
OTC / pink sheet listings excluded

Sector

Exclude financials (GICS 40) and real estate (GICS 60) — standard in cross-sectional return studies, capital structure is fundamentally different
Utilities (GICS 55): this one is worth flagging. Many utilities are ETS-regulated, so they'll get caught by your ETS filter anyway. For non-ETS utilities that slipped through, excluding them is defensible but not mandatory — just be consistent and document it

Fundamentals availability

At least 3 years of non-missing revenue/total assets from your S&P Capital IQ export (you can check this programmatically after the price pull rather than filtering in Orbis)